In [6]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from pypdf import PdfReader
load_dotenv(override=True)

True

In [ ]:

agent = Agent(name="IQ Question Proposer", instructions="You are an IQ Question Proposer", model="gpt-5.4-mini")


In [3]:
result = await Runner.run(agent, "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question.")

print(result.final_output)

A room contains 100 locked boxes arranged in a row, labeled 1 through 100. Each box contains a card with a number on it, and the numbers form a permutation of 1 through 100. You may open any 10 boxes of your choice, but after opening a box you must close it permanently and cannot reopen it. You are then shown the 10 cards you saw, and may make one final guess about whether the original permutation has an even or odd number of inversions. What strategy should you use to maximize your chance of being correct, and what is that maximum probability?


In [4]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Push over user found and looks good")
    else:
        print("Push over user found but doesn't start with u")
else:
    print("Push over user not found")


if pushover_token:
    if pushover_token.startswith("a"):
        print("Push over token found and looks good")
    else:
        print("Push over token found but not start with a")
else:
    print("Push over token not found")

Push over user found and looks good
Push over token found and looks good


In [12]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [13]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [14]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [15]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional info about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],


In [10]:
reader = PdfReader("../1_foundations/twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("../1_foundations/twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [11]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""

In [16]:
session = SQLiteSession("12346")

In [17]:
@function_tool
def chat(message:str) -> str:
    messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": message}]
    notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[tools])
    

In [18]:

result = await Runner.run(notifier, "Notify the user that the pizza is here")
print(result.final_output)

NameError: name 'notifier' is not defined